In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

GEFS_FOLDER = PROJECT / "data" / "raw" / "gefs_pilot"
IMERG_FOLDER = PROJECT / "data" / "raw" / "imerg_pilot"

DAILY_FOLDER = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_daily"
)

JULY_OUTPUT_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_gefs_imerg.nc"
)

DAILY_FOLDER.mkdir(parents=True, exist_ok=True)

In [2]:
gefs_files = sorted(
    GEFS_FOLDER.glob("*.grib2")
)

imerg_files = sorted(
    IMERG_FOLDER.glob("*.nc4")
)


gefs_by_initialization = {}

for file in gefs_files:
    match = re.search(
        r"apcp_sfc_(\d{10})_c00",
        file.name
    )

    if match:
        initialization = pd.to_datetime(
            match.group(1),
            format="%Y%m%d%H"
        )

        gefs_by_initialization[
            initialization
        ] = file


imerg_by_date = {}

for file in imerg_files:
    match = re.search(
        r"3IMERG\.(\d{8})-",
        file.name
    )

    if match:
        observation_date = pd.to_datetime(
            match.group(1),
            format="%Y%m%d"
        )

        imerg_by_date[
            observation_date
        ] = file


print(
    "GEFS files mapped:",
    len(gefs_by_initialization)
)

print(
    "IMERG files mapped:",
    len(imerg_by_date)
)

GEFS files mapped: 31
IMERG files mapped: 31


In [4]:
def process_gefs_file(file):
    """Create +24 to +48-hour GEFS rainfall over India."""

    ds = xr.open_dataset(
        file,
        engine="cfgrib",
        backend_kwargs={
            "filter_by_keys": {
                "typeOfLevel": "surface"
            },
            "indexpath": ""
        }
    )

    try:
        variable_name = (
            "tp"
            if "tp" in ds.data_vars
            else list(ds.data_vars)[0]
        )

        rain = ds[variable_name]

        step_hours = (
            rain.step / np.timedelta64(1, "h")
        ).astype(int)

        selected = rain.where(
            (step_hours > 24)
            & (step_hours <= 48),
            drop=True
        )

        selected_hours = (
            selected.step / np.timedelta64(1, "h")
        ).astype(int).values.tolist()

        expected_hours = [
            27, 30, 33, 36,
            39, 42, 45, 48
        ]

        if selected_hours != expected_hours:
            raise ValueError(
                f"Unexpected forecast steps in "
                f"{file.name}: {selected_hours}"
            )

        rain_24h = selected.sum(
            dim="step",
            skipna=True
        )

        rain_24h = rain_24h.where(
            (rain_24h.latitude >= 6)
            & (rain_24h.latitude <= 38)
            & (rain_24h.longitude >= 68)
            & (rain_24h.longitude <= 98),
            drop=True
        )

        rain_24h = rain_24h.sortby("latitude")
        rain_24h = rain_24h.sortby("longitude")
        rain_24h = rain_24h.load()

    finally:
        ds.close()

    # Keep only the required spatial coordinates.
    cleaned = xr.DataArray(
        rain_24h.values,
        coords={
            "latitude": rain_24h.latitude.values,
            "longitude": rain_24h.longitude.values
        },
        dims=(
            "latitude",
            "longitude"
        ),
        name="gefs_rainfall"
    )

    cleaned.attrs = {
        "long_name": "GEFS 24-hour forecast rainfall",
        "units": "mm",
        "lead_period": "24-48 hours"
    }

    return cleaned

In [5]:
from netCDF4 import Dataset


def process_imerg_file(file, target_grid):
    """Prepare IMERG and interpolate it to the GEFS grid."""

    # Earthaccess files may store variables at root or in Grid.
    with Dataset(file, mode="r") as root:
        groups = list(root.groups.keys())

    open_options = {
        "engine": "netcdf4"
    }

    if "Grid" in groups:
        open_options["group"] = "Grid"

    with xr.open_dataset(
        file,
        **open_options
    ) as ds:
        if "precipitation" not in ds.data_vars:
            raise KeyError(
                f"precipitation not found in {file.name}. "
                f"Variables: {list(ds.data_vars)}"
            )

        rain = ds["precipitation"].load()

    if "time" in rain.dims and rain.sizes["time"] == 1:
        rain = rain.squeeze(
            dim="time",
            drop=True
        )

    rename_map = {}

    if "lat" in rain.dims or "lat" in rain.coords:
        rename_map["lat"] = "latitude"

    if "lon" in rain.dims or "lon" in rain.coords:
        rename_map["lon"] = "longitude"

    rain = rain.rename(rename_map)

    rain = rain.transpose(
        "latitude",
        "longitude"
    )

    rain = rain.sortby("latitude")
    rain = rain.sortby("longitude")

    rain = rain.where(
        (rain.latitude >= 6)
        & (rain.latitude <= 38)
        & (rain.longitude >= 68)
        & (rain.longitude <= 98),
        drop=True
    )

    matched = rain.interp(
        latitude=target_grid.latitude,
        longitude=target_grid.longitude,
        method="linear"
    )

    matched.name = "imerg_rainfall"

    matched.attrs = {
        "long_name": "IMERG rainfall on GEFS grid",
        "units": "mm",
        "original_units": "mm/day",
        "interpolation": "linear"
    }

    return matched

In [6]:
print(
    "GEFS function:",
    callable(process_gefs_file)
)

print(
    "IMERG function:",
    callable(process_imerg_file)
)

GEFS function: True
IMERG function: True


In [7]:
print("GEFS function:", callable(process_gefs_file))
print("IMERG function:", callable(process_imerg_file))

GEFS function: True
IMERG function: True


In [8]:
observation_dates = pd.date_range(
    start="2018-07-01",
    end="2018-07-31",
    freq="D"
)

processing_results = []


for observation_date in observation_dates:
    initialization_date = (
        observation_date
        - pd.Timedelta(days=1)
    )

    daily_file = (
        DAILY_FOLDER
        / f"paired_{observation_date:%Y%m%d}.nc"
    )

    # Reuse successfully processed dates.
    if daily_file.exists() and daily_file.stat().st_size > 0:
        print(
            "Already processed:",
            observation_date.date()
        )

        processing_results.append({
            "date": observation_date.date(),
            "success": True,
            "status": "existing"
        })

        continue

    try:
        if initialization_date not in gefs_by_initialization:
            raise FileNotFoundError(
                f"Missing GEFS initialization "
                f"{initialization_date.date()}"
            )

        if observation_date not in imerg_by_date:
            raise FileNotFoundError(
                f"Missing IMERG date "
                f"{observation_date.date()}"
            )

        gefs_rain = process_gefs_file(
            gefs_by_initialization[
                initialization_date
            ]
        )

        imerg_rain = process_imerg_file(
            imerg_by_date[
                observation_date
            ],
            gefs_rain
        )

        forecast_error = (
            gefs_rain - imerg_rain
        )

        forecast_error.name = "forecast_error"
        forecast_error.attrs["units"] = "mm"

        daily_ds = xr.Dataset({
            "gefs_rainfall": gefs_rain,
            "imerg_rainfall": imerg_rain,
            "forecast_error": forecast_error
        })

        daily_ds = daily_ds.expand_dims(
            date=[
                observation_date.to_datetime64()
            ]
        )

        daily_ds.attrs = {
            "observation_date": str(
                observation_date.date()
            ),
            "gefs_initialization": str(
                initialization_date.date()
            ),
            "forecast_lead": "24-48 hours"
        }

        temporary_file = daily_file.with_suffix(
            ".tmp.nc"
        )

        daily_ds.to_netcdf(
            temporary_file,
            mode="w",
            engine="netcdf4"
        )

        temporary_file.replace(daily_file)

        processing_results.append({
            "date": observation_date.date(),
            "success": True,
            "status": "processed"
        })

        print(
            "Processed:",
            observation_date.date()
        )

    except Exception as error:
        processing_results.append({
            "date": observation_date.date(),
            "success": False,
            "status": str(error)
        })

        print(
            "Failed:",
            observation_date.date(),
            "|",
            error
        )

Processed: 2018-07-01
Processed: 2018-07-02
Processed: 2018-07-03
Processed: 2018-07-04
Processed: 2018-07-05
Processed: 2018-07-06
Processed: 2018-07-07
Processed: 2018-07-08
Processed: 2018-07-09
Processed: 2018-07-10
Processed: 2018-07-11
Processed: 2018-07-12
Processed: 2018-07-13
Processed: 2018-07-14
Processed: 2018-07-15
Processed: 2018-07-16
Processed: 2018-07-17
Processed: 2018-07-18
Processed: 2018-07-19
Processed: 2018-07-20
Processed: 2018-07-21
Processed: 2018-07-22
Processed: 2018-07-23
Processed: 2018-07-24
Processed: 2018-07-25
Processed: 2018-07-26
Processed: 2018-07-27
Processed: 2018-07-28
Processed: 2018-07-29
Processed: 2018-07-30
Processed: 2018-07-31


In [9]:
processing_df = pd.DataFrame(
    processing_results
)

display(processing_df)

successful_dates = int(
    processing_df["success"].sum()
)

print(
    "Successful dates:",
    successful_dates,
    "/ 31"
)

,date,success,status
0,2018-07-01,True,processed
1,2018-07-02,True,processed
2,2018-07-03,True,processed
3,2018-07-04,True,processed
4,2018-07-05,True,processed
5,2018-07-06,True,processed
6,2018-07-07,True,processed
7,2018-07-08,True,processed
8,2018-07-09,True,processed
9,2018-07-10,True,processed


Successful dates: 31 / 31


In [10]:
daily_files = sorted(
    DAILY_FOLDER.glob("paired_*.nc")
)

print(
    "Daily paired files:",
    len(daily_files)
)

for file in daily_files:
    print(
        file.name,
        f"{file.stat().st_size / 1_000_000:.2f} MB"
    )

Daily paired files: 31
paired_20180701.nc 0.32 MB
paired_20180702.nc 0.32 MB
paired_20180703.nc 0.32 MB
paired_20180704.nc 0.32 MB
paired_20180705.nc 0.32 MB
paired_20180706.nc 0.32 MB
paired_20180707.nc 0.32 MB
paired_20180708.nc 0.32 MB
paired_20180709.nc 0.32 MB
paired_20180710.nc 0.32 MB
paired_20180711.nc 0.32 MB
paired_20180712.nc 0.32 MB
paired_20180713.nc 0.32 MB
paired_20180714.nc 0.32 MB
paired_20180715.nc 0.32 MB
paired_20180716.nc 0.32 MB
paired_20180717.nc 0.32 MB
paired_20180718.nc 0.32 MB
paired_20180719.nc 0.32 MB
paired_20180720.nc 0.32 MB
paired_20180721.nc 0.32 MB
paired_20180722.nc 0.32 MB
paired_20180723.nc 0.32 MB
paired_20180724.nc 0.32 MB
paired_20180725.nc 0.32 MB
paired_20180726.nc 0.32 MB
paired_20180727.nc 0.32 MB
paired_20180728.nc 0.32 MB
paired_20180729.nc 0.32 MB
paired_20180730.nc 0.32 MB
paired_20180731.nc 0.32 MB


In [11]:
if len(daily_files) != 31:
    raise ValueError(
        "Expected 31 daily files before combining."
    )


daily_datasets = []

for file in daily_files:
    with xr.open_dataset(file) as ds:
        daily_datasets.append(
            ds.load()
        )


july_ds = xr.concat(
    daily_datasets,
    dim="date"
)

july_ds = july_ds.sortby("date")

july_ds.attrs = {
    "title": "July 2018 GEFS and IMERG rainfall dataset",
    "period": "2018-07-01 to 2018-07-31",
    "forecast_source": "NOAA GEFSv12 reforecast c00",
    "observation_source": "NASA GPM IMERG Final V07",
    "forecast_lead": "24-48 hours",
    "error_definition": "GEFS minus IMERG"
}

print(july_ds)

<xarray.Dataset> Size: 10MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    gefs_rainfall   (date, latitude, longitude) float32 2MB 1.91 1.21 ... 1.35
    imerg_rainfall  (date, latitude, longitude) float64 4MB nan nan ... nan nan
    forecast_error  (date, latitude, longitude) float64 4MB nan nan ... nan nan
Attributes:
    title:               July 2018 GEFS and IMERG rainfall dataset
    period:              2018-07-01 to 2018-07-31
    forecast_source:     NOAA GEFSv12 reforecast c00
    observation_source:  NASA GPM IMERG Final V07
    forecast_lead:       24-48 hours
    error_definition:    GEFS minus IMERG


In [12]:
expected_dates = pd.date_range(
    "2018-07-01",
    "2018-07-31",
    freq="D"
)

actual_dates = pd.to_datetime(
    july_ds.date.values
)

missing_dates = expected_dates.difference(
    actual_dates
)

duplicate_dates = actual_dates[
    actual_dates.duplicated()
]


print("Number of dates:", july_ds.sizes["date"])
print("Missing dates:", list(missing_dates))
print("Duplicate dates:", list(duplicate_dates))

for variable in [
    "gefs_rainfall",
    "imerg_rainfall",
    "forecast_error"
]:
    missing_count = int(
        july_ds[variable].isnull().sum()
    )

    print(
        variable,
        "| missing values:",
        missing_count
    )

Number of dates: 31
Missing dates: []
Duplicate dates: []
gefs_rainfall | missing values: 0
imerg_rainfall | missing values: 15376
forecast_error | missing values: 15376


In [13]:
summary_rows = []


for date in july_ds.date.values:
    daily = july_ds.sel(date=date)

    summary_rows.append({
        "date": pd.Timestamp(date).date(),
        "gefs_mean": float(
            daily["gefs_rainfall"].mean(
                skipna=True
            )
        ),
        "imerg_mean": float(
            daily["imerg_rainfall"].mean(
                skipna=True
            )
        ),
        "bias": float(
            daily["forecast_error"].mean(
                skipna=True
            )
        ),
        "gefs_max": float(
            daily["gefs_rainfall"].max(
                skipna=True
            )
        ),
        "imerg_max": float(
            daily["imerg_rainfall"].max(
                skipna=True
            )
        )
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)

,date,gefs_mean,imerg_mean,bias,gefs_max,imerg_max
0,2018-07-01,11.300482,6.585499,4.841639,335.600006,151.800003
1,2018-07-02,13.650534,7.904381,5.954987,401.179993,156.095001
2,2018-07-03,13.486231,7.606247,6.050426,188.250000,273.917496
3,2018-07-04,13.330412,8.198208,5.258902,265.780029,254.345001
4,2018-07-05,12.204255,7.915879,4.377851,167.809998,253.055023
5,2018-07-06,11.245247,7.546630,3.664125,143.199982,249.272507
6,2018-07-07,11.687757,6.580757,5.269948,130.100006,168.705009
7,2018-07-08,11.137304,6.278675,5.000705,171.200012,182.395004
8,2018-07-09,12.702113,7.640871,5.222303,186.279999,192.075012
9,2018-07-10,15.414711,9.772401,5.845764,169.379990,162.370010


In [14]:
encoding = {
    variable: {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    }
    for variable in july_ds.data_vars
}


july_ds.to_netcdf(
    JULY_OUTPUT_FILE,
    mode="w",
    engine="netcdf4",
    encoding=encoding
)


print(
    "Saved:",
    JULY_OUTPUT_FILE.exists()
)

print(
    "Location:",
    JULY_OUTPUT_FILE
)

print(
    "Size:",
    JULY_OUTPUT_FILE.stat().st_size
    / 1_000_000,
    "MB"
)

Saved: True
Location: Z:\Projects\monsoon-postprocessing\data\processed\july2018_gefs_imerg.nc
Size: 3.714329 MB


In [15]:
with xr.open_dataset(
    JULY_OUTPUT_FILE
) as saved_ds:

    print(saved_ds)

    print(
        "Saved dates:",
        saved_ds.sizes["date"]
    )

    print(
        "Variables:",
        list(saved_ds.data_vars)
    )

<xarray.Dataset> Size: 6MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    gefs_rainfall   (date, latitude, longitude) float32 2MB ...
    imerg_rainfall  (date, latitude, longitude) float32 2MB ...
    forecast_error  (date, latitude, longitude) float32 2MB ...
Attributes:
    title:               July 2018 GEFS and IMERG rainfall dataset
    period:              2018-07-01 to 2018-07-31
    forecast_source:     NOAA GEFSv12 reforecast c00
    observation_source:  NASA GPM IMERG Final V07
    forecast_lead:       24-48 hours
    error_definition:    GEFS minus IMERG
Saved dates: 31
Variables: ['gefs_rainfall', 'imerg_rainfall', 'forecast_error']
